In [ ]:
%matplotlib ipympl

In [ ]:
import functools

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.interpolate as sci_interp
import scipy.optimize as sci_opt
from helper import *

jax.config.update("jax_enable_x64", True)

## data filter helpers

In [ ]:
@functools.partial(jax.jit, static_argnames=["nu"])
def get_E0_C(den, num, K=None, nu=0):
    """Integration ZOH scheme."""
    if K is None:
        K = jnp.prod(den) / jnp.prod(num)
    a = jnp.poly(den)
    b = K * jnp.poly(num)
    b = jnp.concatenate([jnp.atleast_1d(b), jnp.zeros(nu)])
    assert a.size - b.size >= 1, f"(a, b) = ({a.size}, {b.size})"

    a_coeffs = a[1:]
    n = a_coeffs.size
    A = jnp.vstack([-a_coeffs, jnp.eye(n - 1, n)])
    C = jnp.concatenate([jnp.zeros(n - b.size), b])
    E0 = jax.scipy.linalg.expm(A * dt)
    return E0, C

In [ ]:
def _mpow_grad_base(base: jax.Array, exp: jax.Array, cot: jax.Array) -> jax.Array:
    bt = base.T

    def body(i, acc):
        left = mpow(bt, i)
        right = mpow(bt, exp - 1 - i)
        return acc + left @ cot @ right

    return jax.lax.cond(
        exp == 0,
        lambda: jnp.zeros_like(base),
        lambda: jax.lax.fori_loop(
            0, exp, body, jnp.zeros_like(base)
        ),
    )


def _mpow_with_custom_vjp(fun):
    wrapped = jax.custom_vjp(fun)

    def fwd(A, n):
        y = fun(A, n)
        n_i64 = jnp.asarray(n, dtype=jnp.int64)
        base = jax.lax.cond(n_i64 < 0, lambda: jnp.linalg.inv(A), lambda: A)
        exp = jnp.abs(n_i64)
        return y, (A, base, exp, n_i64)

    def bwd(res, g):
        A, base, exp, n_i64 = res
        grad_base = _mpow_grad_base(base, exp, g)
        grad_A = jax.lax.cond(
            n_i64 < 0,
            lambda: -(base.T @ grad_base @ base.T),
            lambda: grad_base,
        )
        return grad_A, None

    wrapped.defvjp(fwd, bwd)
    return wrapped


@_mpow_with_custom_vjp
def mpow(A: jax.Array, n: jax.Array) -> jax.Array:
    n = jnp.asarray(n, dtype=jnp.int64)

    base = jax.lax.cond(
        n < 0,
        lambda: jnp.linalg.inv(A),
        lambda: A,
    )
    exp = jnp.abs(n)

    result0 = jnp.eye(A.shape[0], dtype=A.dtype)

    def cond_fn(state):
        _, k, _ = state
        return k > 0

    def body_fn(state):
        result, k, b = state
        result = jax.lax.cond(
            (k & 1) == 1,
            lambda r: r @ b,
            lambda r: r,
            result,
        )
        return result, k // 2, b @ b

    result, _, _ = jax.lax.while_loop(cond_fn, body_fn, (result0, exp, base))
    return result

# np.random.seed(42)
# A = np.random.uniform(-1.0, 1.0, size=4**2).reshape(4, 4)
# for i in range(10):
#     diff = jax.jacrev(mpow)(A, i) - jax.jacrev(jnp.linalg.matrix_power)(A, i)
#     print(i)
#     assert jnp.allclose(diff, 0.0, atol=1e-13, rtol=0.0)

In [ ]:
@jax.jit
def obs_x0(A, B, C, D, y, u) -> np.ndarray:
    n = A.shape[0]
    squee = jnp.squeeze
    cond = jax.lax.cond

    y = jnp.ravel(y)
    u = jnp.ravel(u)
    assert y.size == n
    assert u.size == n

    if n == 1:
        return jnp.atleast_1d((y - squee(D) * u) / squee(C))

    O = jnp.vstack([C @ mpow(A, i) for i in range(n)])

    def U_fun(i, j):
        return cond(
                (i == 0) & (j == 0),
                lambda: squee(D),
                lambda: cond(
                    ((i == 0) & (j != 0)) | ((i != 0) & (j == 0)) | (j > i),
                    lambda: 0.0,
                    lambda: cond(
                        i == j,
                        lambda: squee(C @ B + D),
                        lambda: squee(C @ mpow(A, i - j) @ B)
                    )
                )
        )

    U = jnp.fromfunction(jnp.vectorize(U_fun), (n, n), dtype=int)
    return jnp.linalg.solve(O, y - U @ u)

In [ ]:
@functools.partial(jax.jit, static_argnames=["n"])
def linear_pred(A, C, x0, n):
    res = jnp.empty(n)
    res = res.at[0].set(C @ x0)

    def pred_body(i, state):
        res, x0 = state
        x1 = A @ x0
        res = res.at[i].set(C @ x1)
        return (res, x1)

    res, _ = jax.lax.fori_loop(1, n, pred_body, (res, x0))
    return res

@functools.partial(jax.jit, static_argnames=["n"])
def linear_pred_hist(A, C, hist, n):
    n_A = A.shape[0]
    assert hist.size == n_A
    x0 = obs_x0(A, jnp.zeros(n_A), C, jnp.zeros(1), hist, jnp.zeros(n_A))
    x0 = mpow(A, n_A - 1) @ x0
    return linear_pred(A, C, x0, n)

In [ ]:
def pred_err(pred_fun, data, hist_size):

    def err_body(hist, future, idx):
        pred = pred_fun(hist)
        return jnp.mean(jnp.square((pred - future) * 1e2))

    hist = part(data, hist_size)
    future = part(data[hist_size:], spec.n)
    hist = hist[: future.shape[0]]
    idx = jnp.arange(hist_size - 1, hist_size - 1 + hist.shape[0])
    err = jax.vmap(err_body)(hist, future, idx)
    return jnp.mean(err)

@jax.jit
def pred_cost(den, num, data):
    A, C = get_E0_C(den, num)

    def pred_fun(hist):
        assert hist.size == A.shape[0]
        return linear_pred_hist(A, C, hist, spec.n)

    return pred_err(pred_fun, data, A.shape[0])

In [ ]:
_, _, data, _ = get_data(0)

den_size = 4

def sci_fun(arr):
    assert arr.size >= den_size and arr.size <= den_size * 2 - 1
    return pred_cost(arr[:den_size], arr[den_size:], data)

sci_fun_grad = jax.jit(jax.value_and_grad(sci_fun))

In [ ]:
x0 = np.ones(5) * -1e1
sci_fun_grad(x0)

In [ ]:
size = 6
x0 = np.ones(size) * -1e1 + np.random.uniform(-2.0, 2.0, size)
res = sci_opt.minimize(
    fun=sci_fun_grad,
    x0=x0,
    method="L-BFGS-B",
    jac=True,
    # bounds=[(-np.inf, 0)] * x0.size,
)
res

In [ ]:
idx = 4000

A, C = get_E0_C(res.x[:den_size], res.x[den_size:])
pred = linear_pred_hist(A, C, data[idx: idx + den_size], 200)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(data[idx + den_size - 1: idx + 200 + den_size - 1], label="data")
ax.plot(pred, label="pred")
ax.grid()
ax.legend()